In [1]:
from helper import *

I moved classes and functions from the previous notebook to the `helper.py` not to rewrite the same code

In [2]:
HR_train_paths = sorted(glob.glob("../data/DIV2K_train_HR/*.png"))
X2_train_paths = sorted(glob.glob("../data/DIV2K_train_LR_bicubic/X2/*.png"))
X4_train_paths = sorted(glob.glob("../data/DIV2K_train_LR_bicubic/X4/*.png"))
X8_train_paths = sorted(glob.glob("../data/DIV2K_train_LR_bicubic/X8/*.png"))
X16_train_paths = sorted(glob.glob("../data/DIV2K_train_LR_bicubic/X16/*.png"))
X32_train_paths = sorted(glob.glob("../data/DIV2K_train_LR_bicubic/X32/*.png"))
X64_train_paths = sorted(glob.glob("../data/DIV2K_train_LR_bicubic/X64/*.png"))

HR_valid_paths = sorted(glob.glob("../data/DIV2K_valid_HR/*.png"))
X2_valid_paths = sorted(glob.glob("../data/DIV2K_valid_LR_bicubic/X2/*.png"))
X4_valid_paths = sorted(glob.glob("../data/DIV2K_valid_LR_bicubic/X4/*.png"))
X8_valid_paths = sorted(glob.glob("../data/DIV2K_valid_LR_bicubic/X8/*.png"))
X16_valid_paths = sorted(glob.glob("../data/DIV2K_valid_LR_bicubic/X16/*.png"))
X32_valid_paths = sorted(glob.glob("../data/DIV2K_valid_LR_bicubic/X32/*.png"))
X64_valid_paths = sorted(glob.glob("../data/DIV2K_valid_LR_bicubic/X64/*.png"))

## X8 Scaling

The process of preparation for training is the same as in 4X notebook, but the starting point is the **4X model**, not the 2X model

In [4]:
checkpoint = torch.load('../model_checkpoints/SRResNet/X4.pth')
model = SRResNet(4).to(device)
model.load_state_dict(checkpoint['model_state_dict'])

model.upscaling_head = model.upscaling_head[:-1]

head = nn.Sequential(
    nn.Conv2d(64, 256, 3, stride=1, padding='same'),
    nn.PixelShuffle(2),
    nn.PReLU(),
    nn.Conv2d(64, 3, 9, stride=1, padding='same')
)

model.upscaling_head = nn.Sequential(*model.upscaling_head, *head)

for param in model.expand.parameters():
    param.requires_grad = False

for param in model.residual_blocks.parameters():
    param.requires_grad = False

for param in model.upscaling_head[:-4].parameters():
    param.requires_grad = False

summary(model, col_names=['trainable'])

Layer (type:depth-idx)                   Trainable
SRResNet                                 Partial
├─Sequential: 1-1                        False
│    └─Conv2d: 2-1                       False
│    └─PReLU: 2-2                        False
├─Sequential: 1-2                        False
│    └─ResBlock: 2-3                     False
│    │    └─Sequential: 3-1              False
│    └─ResBlock: 2-4                     False
│    │    └─Sequential: 3-2              False
│    └─ResBlock: 2-5                     False
│    │    └─Sequential: 3-3              False
│    └─ResBlock: 2-6                     False
│    │    └─Sequential: 3-4              False
│    └─ResBlock: 2-7                     False
│    │    └─Sequential: 3-5              False
│    └─ResBlock: 2-8                     False
│    │    └─Sequential: 3-6              False
│    └─ResBlock: 2-9                     False
│    │    └─Sequential: 3-7              False
│    └─ResBlock: 2-10                    False
│    │ 

In [5]:
valid_ds = SRResNet_Dataset(HR_valid_paths, 8, ram_limit_gb=1)

Preloading images:   0%|          | 0/100 [00:00<?, ?it/s]

In [6]:
train_ds = SRResNet_Dataset(HR_train_paths, 8, ram_limit_gb=8)

Preloading images:   0%|          | 0/800 [00:00<?, ?it/s]

In [8]:
train_dl = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=os.cpu_count()-1)
valid_dl = DataLoader(valid_ds, batch_size=16, shuffle=False, num_workers=os.cpu_count()-1)

loss_fn = nn.L1Loss()
optimizer = Adam(model.upscaling_head[-4:].parameters(), lr=1e-4)
scheduler = StepLR(optimizer, step_size=200, gamma=0.5)
model = model.to(device) # put on device the new upscaling head

train(model, train_dl, valid_dl, optimizer, scheduler, loss_fn, 1000)

Epochs:   0%|          | 0/1000 [00:00<?, ?it/s]

Epoch: 1 | learning rate: 0.0001000 | [train] PSNR: 14.7026 | [train] SSIM: 0.4425 | [valid] PSNR: 18.7048 | [valid] SSIM: 0.5042 | [valid] LPIPS: 0.6520
Epoch: 2 | learning rate: 0.0001000 | [train] PSNR: 20.3242 | [train] SSIM: 0.5379 | [valid] PSNR: 21.3682 | [valid] SSIM: 0.5557 | [valid] LPIPS: 0.5796
Epoch: 3 | learning rate: 0.0001000 | [train] PSNR: 21.7445 | [train] SSIM: 0.5688 | [valid] PSNR: 21.9675 | [valid] SSIM: 0.5801 | [valid] LPIPS: 0.5603
Epoch: 4 | learning rate: 0.0001000 | [train] PSNR: 22.3808 | [train] SSIM: 0.5946 | [valid] PSNR: 22.1608 | [valid] SSIM: 0.5819 | [valid] LPIPS: 0.5452
Epoch: 5 | learning rate: 0.0001000 | [train] PSNR: 22.4554 | [train] SSIM: 0.5973 | [valid] PSNR: 22.5049 | [valid] SSIM: 0.5942 | [valid] LPIPS: 0.5335
Epoch: 6 | learning rate: 0.0001000 | [train] PSNR: 22.6029 | [train] SSIM: 0.6012 | [valid] PSNR: 22.9958 | [valid] SSIM: 0.6146 | [valid] LPIPS: 0.5225


KeyboardInterrupt: 

In [ ]:
checkpoint = torch.load('../tmp_model_checkpoints/last.pth')
model = SRResNet(8).to(device)
model.load_state_dict(checkpoint['model_state_dict'])

In [ ]:
model.eval();

In [ ]:
targets = [X8_valid_paths, X4_valid_paths, X2_valid_paths, HR_valid_paths]

metrics_x8 = pd.DataFrame(columns=["PSNR↑", "SSIM↑", "LPIPS↓"])
for target_ds in tqdm(targets, total=4):
    metrics_x8.loc[len(metrics_x8)] = calc_metrics(model, target_ds, 8)

metrics_x8.index = ["31px -> 248px", "63px -> 504px", "127px -> 1016px", "255px -> 2040px"]
metrics_x8

In [ ]:
train_dl = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=os.cpu_count()-1)
valid_dl = DataLoader(valid_ds, batch_size=16, shuffle=False, num_workers=os.cpu_count()-1)

loss_fn = nn.L1Loss()
optimizer = Adam(model.parameters(), lr=1e-5)
scheduler = StepLR(optimizer, step_size=1000, gamma=0.5)

train(model, train_dl, valid_dl, optimizer, scheduler, loss_fn, 5000)

In [ ]:
checkpoint = torch.load('../model_checkpoints/SRResNet/X8.pth')
model = SRResNet(8).to(device)
model.load_state_dict(checkpoint['model_state_dict'])

In [ ]:
model.eval();

In [ ]:
targets = [X8_valid_paths, X4_valid_paths, X2_valid_paths, HR_valid_paths]

metrics_x8 = pd.DataFrame(columns=["PSNR↑", "SSIM↑", "LPIPS↓"])
for target_ds in tqdm(targets, total=4):
    metrics_x8.loc[len(metrics_x8)] = calc_metrics(model, target_ds, 8)

metrics_x8.index = ["31px -> 248px", "63px -> 504px", "127px -> 1016px", "255px -> 2040px"]
metrics_x8

### Super-resolution showcase

In [ ]:
print("31px -> 248px")
inp = transform(Image.open(X64_valid_paths[4])).to(device)
with torch.inference_mode():
    out = (((model(inp[None])+1.0)/2.0).clamp(0.0, 1.0) * 255.0).squeeze()

img = Image.fromarray(out.permute(1, 2, 0).to(torch.uint8).cpu().numpy())
os.makedirs('../image_results/8X/31px', exist_ok=True)
img.save('../image_results/8X/31px/SRResNet_31px.png')
img

In [ ]:
print("63px -> 504px")
inp = transform(Image.open(X32_valid_paths[4])).to(device)
with torch.inference_mode():
    out = (((model(inp[None])+1.0)/2.0).clamp(0.0, 1.0) * 255.0).squeeze()

img = Image.fromarray(out.permute(1, 2, 0).to(torch.uint8).cpu().numpy())
os.makedirs('../image_results/8X/63px', exist_ok=True)
img.save('../image_results/8X/63px/SRResNet_63px.png')
img

In [ ]:
print("127px -> 1016px")
inp = transform(Image.open(X16_valid_paths[4])).to(device)
with torch.inference_mode():
    out = (((model(inp[None])+1.0)/2.0).clamp(0.0, 1.0) * 255.0).squeeze()

img = Image.fromarray(out.permute(1, 2, 0).to(torch.uint8).cpu().numpy())
os.makedirs('../image_results/8X/127px', exist_ok=True)
img.save('../image_results/8X/127px/SRResNet_127px.png')
img

In [ ]:
print("255px -> 2040px")
inp = transform(Image.open(X8_valid_paths[4])).to(device)
with torch.inference_mode():
    out = (((model(inp[None])+1.0)/2.0).clamp(0.0, 1.0) * 255.0).squeeze()

img = Image.fromarray(out.permute(1, 2, 0).to(torch.uint8).cpu().numpy())
os.makedirs('../image_results/8X/255px', exist_ok=True)
img.save('../image_results/8X/255px/SRResNet_255px.png')
img